# 🏥 CareAgent: Clinical Risk Backbone Machine Learning Pipeline

This notebook demonstrates how to train a **Clinical Risk Backbone** classifier (Stage 1 of the CareAgent architecture) using purely clinical encounter and claims data (excluding SDOH surveys).

### Pipeline Overview:
1. **Data Ingestion**: Load and merge patient demographics and encounters data.
2. **Feature Engineering**: Filter features down to purely clinical parameters.
3. **Preprocessing**: Apply categorical label encoding.
4. **Leakage-Free Validation Split**: Segment training and testing sets on patient-level IDs.
5. **Ensemble Model Training**: Fit Random Forest classifiers for 30, 60, and 90-day readmission risk horizons.
6. **Evaluation & Performance Analysis**: Calculate accuracy, ROC-AUC, and plot feature importances.
7. **Inference Demo**: Execute a mock prediction function on a new encounter.

## Step 1: Environment Setup & Imports
We load standard scientific Python libraries required for data manipulation, machine learning, and plotting.

In [ ]:
import os
import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style="darkgrid")

## Step 2: Data Ingestion & Merging
We load the synthetic patient demographic and encounter data files from the `data/` folder and merge them using `patient_id`.

In [ ]:
data_dir = "data"
patients_df = pd.read_csv(os.path.join(data_dir, "careagent_patients_5000.csv"))
encounters_df = pd.read_csv(os.path.join(data_dir, "careagent_encounters_5000.csv"))

print(f"Loaded {len(patients_df)} patient profiles and {len(encounters_df)} clinical encounters.")

# Merge datasets on patient_id
df = encounters_df.merge(patients_df, on="patient_id", how="left")
print(f"Joined dataset contains {len(df)} rows and {df.shape[1]} columns.")

## Step 3: Feature Definition & Selection
Here, we restrict our features **strictly to clinical and encounter data**, completely ignoring any SDOH metrics (such as food insecurity or income barriers). This isolates the clinical factors for our backbone model.

In [ ]:
# Clinical features
categorical_cols = ["sex", "insurance", "language", "encounter_type", "diagnosis_group"]
numerical_cols = ["age", "length_of_stay", "prior_encounters", "prior_ed", "prior_inpatient"]

feature_cols = categorical_cols + numerical_cols

print("Selected Clinical Features:")
for idx, col in enumerate(feature_cols):
    print(f"{idx+1}. {col}")

## Step 4: Categorical Label Encoding
We fit a separate `LabelEncoder` for each categorical column to convert strings into integer values, preserving the mappings to handle unseen categories during inference.

In [ ]:
encoders = {}
df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f"Encoded '{col}' into {len(le.classes_)} classes.")

## Step 5: Leakage-Free Train-Test Split
If a patient has multiple encounters, splitting purely at the encounter level will leak their demographics and history across train and test sets. To prevent this, we perform a **Patient-Level Split** by partitioning unique patient IDs first, then segmenting the encounters.

In [ ]:
unique_patients = df_encoded["patient_id"].unique()
train_patients, test_patients = train_test_split(unique_patients, test_size=0.2, random_state=42)

train_df = df_encoded[df_encoded["patient_id"].isin(train_patients)]
test_df = df_encoded[df_encoded["patient_id"].isin(test_patients)]

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

# Readmission targets for 30, 60, and 90 days
y_train_30, y_test_30 = train_df["readmit_30"], test_df["readmit_30"]
y_train_60, y_test_60 = train_df["readmit_60"], test_df["readmit_60"]
y_train_90, y_test_90 = train_df["readmit_90"], test_df["readmit_90"]

print(f"Training set size: {len(X_train)} encounters")
print(f"Testing set size: {len(X_test)} encounters")

## Step 6: Model Training
We train three separate `RandomForestClassifier` models. Random Forest is highly robust, handles non-linear relationships, and does not require feature scaling.

In [ ]:
print("Training 30-Day Risk Classifier...")
rf_30 = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_30.fit(X_train, y_train_30)

print("Training 60-Day Risk Classifier...")
rf_60 = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_60.fit(X_train, y_train_60)

print("Training 90-Day Risk Classifier...")
rf_90 = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_90.fit(X_train, y_train_90)

print("All models trained successfully!")

## Step 7: Model Evaluation
We predict probabilities and calculate accuracy and Area Under the ROC Curve (ROC-AUC) for all three horizons.

In [ ]:
def evaluate_model(model, X, y, label):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    acc = accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    print(f"=== {label} Evaluation ===")
    print(f"Accuracy: {acc:.4f}")
    print(f"ROC-AUC:  {auc:.4f}\n")
    print(classification_report(y, preds))

evaluate_model(rf_30, X_test, y_test_30, "30-Day Risk")
evaluate_model(rf_60, X_test, y_test_60, "60-Day Risk")
evaluate_model(rf_90, X_test, y_test_90, "90-Day Risk")

## Step 8: Global Feature Importance Analysis
Let's see which clinical features contribute most to the 30-Day Readmission risk model.

In [ ]:
importances = rf_30.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(10, 6))
plt.title("Clinical Risk Backbone: Feature Importances (30-Day)", fontsize=14, fontweight="bold")
plt.barh(range(len(indices)), importances[indices], color="#2a5298", align="center")
plt.yticks(range(len(indices)), [feature_cols[i] for i in indices])
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

## Step 9: Offline Inference Demonstration
We write a wrapper function demonstrating how to intake a new encounter event and generate low/medium/high risk categorizations on-demand.

In [ ]:
def predict_patient_readmission_risk(patient_encounter):
    """
    Runs offline inference for a single patient encounter.
    """
    # Convert single dictionary to DataFrame
    sample_df = pd.DataFrame([patient_encounter])
    
    # Apply encoders
    for col in categorical_cols:
        le = encoders[col]
        val = str(sample_df[col].iloc[0])
        if val in le.classes_:
            sample_df[col] = le.transform([val])[0]
        else:
            sample_df[col] = 0
            
    # Select features
    X = sample_df[feature_cols]
    
    # Predict probabilities
    prob_30 = rf_30.predict_proba(X)[0][1]
    prob_60 = rf_60.predict_proba(X)[0][1]
    prob_90 = rf_90.predict_proba(X)[0][1]
    
    # Determine risk band thresholds (Low < 15%, Medium < 35%, High otherwise)
    def get_band(p):
        if p < 0.15: return "Low"
        if p < 0.35: return "Medium"
        return "High"
        
    return {
        "30_day_risk": {"probability": f"{prob_30:.1%}", "band": get_band(prob_30)},
        "60_day_risk": {"probability": f"{prob_60:.1%}", "band": get_band(prob_60)},
        "90_day_risk": {"probability": f"{prob_90:.1%}", "band": get_band(prob_90)}
    }

# Mock patient encounter data (without SDOH)
mock_encounter = {
    "sex": "F",
    "insurance": "Medicare",
    "language": "English",
    "encounter_type": "Inpatient",
    "diagnosis_group": "CHF",
    "age": 74,
    "length_of_stay": 6,
    "prior_encounters": 2,
    "prior_ed": 1,
    "prior_inpatient": 1
}

results = predict_patient_readmission_risk(mock_encounter)
print("Predicted Risks for Mock Encounter:")
print(json.dumps(results, indent=2))